# 03 — Algorithm Comparison: The Arena

Run all six detectors on all three datasets. Report precision, recall, F1 at the true contamination rate, and AUPRC. Produce the arena scoreboard.

In [1]:
import sys
from pathlib import Path

# Ensure project root is on sys.path regardless of cwd
_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import json
import warnings

warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FIGURES = _ROOT / "outputs" / "figures"
OUTPUTS = _ROOT / "outputs"
FIGURES.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

from src.datasets import load_network, load_manufacturing, make_synthetic_credit_card
from src.detectors import (
    IsolationForestDetector,
    LOFDetector,
    OneClassSVMDetector,
    DBSCANDetector,
    MahalanobisDetector,
    AutoencoderDetector,
)
from src.evaluation import (
    run_arena,
    sweep_contamination,
    evaluate_detector,
    auprc,
    precision_recall_f1_at_contamination,
)
import src.visualisation as vis


## Load datasets

In [2]:
X_cc, y_cc = make_synthetic_credit_card(n_samples=5000, random_state=42)
X_net, y_net = load_network(n_samples=3000, random_state=42)
X_mfg, y_mfg = load_manufacturing(n_samples=2000, random_state=42)
print("Loaded.")


Loaded.


## Run the arena

In [3]:
# Use fewer autoencoder epochs for notebook speed; increase for production
detectors = [
    IsolationForestDetector(n_estimators=100, random_state=42),
    LOFDetector(n_neighbors=20),
    OneClassSVMDetector(nu=0.05, max_samples=2000),
    DBSCANDetector(eps=0.5, min_samples=5),
    MahalanobisDetector(),
    AutoencoderDetector(hidden_dim=32, epochs=20, random_state=42),
]

contamination = {"credit_card": 0.0017, "network": 0.02, "manufacturing": 0.015}

datasets_arena = {
    "credit_card": (X_cc, y_cc, contamination["credit_card"]),
    "network": (X_net, y_net, contamination["network"]),
    "manufacturing": (X_mfg, y_mfg, contamination["manufacturing"]),
}

print("Running arena (this takes ~1–2 minutes)...")
scoreboard = run_arena(detectors, datasets_arena)
print("Done.")
scoreboard.round(3)


Running arena (this takes ~1–2 minutes)...


Done.


,name,precision,recall,f1,auprc,dataset
0,IsolationForest,0.500,1.000,0.667,1.000,credit_card
1,LOF,0.500,1.000,0.667,1.000,credit_card
2,OneClassSVM,0.500,1.000,0.667,1.000,credit_card
3,DBSCAN,0.500,1.000,0.667,1.000,credit_card
4,Mahalanobis,0.500,1.000,0.667,1.000,credit_card
5,Autoencoder,0.000,0.000,0.000,0.043,credit_card
6,IsolationForest,0.833,1.000,0.909,1.000,network
7,LOF,0.250,0.300,0.273,0.297,network
8,OneClassSVM,0.250,0.300,0.273,0.251,network
9,DBSCAN,0.167,0.200,0.182,0.103,network


## Scoreboard heatmap — AUPRC

In [4]:
fig = vis.plot_arena_scoreboard(scoreboard, metric="auprc")
fig.savefig(FIGURES / "03_arena_auprc_heatmap.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved 03_arena_auprc_heatmap.png")


Saved 03_arena_auprc_heatmap.png


## Scoreboard heatmap — F1

In [5]:
fig = vis.plot_arena_scoreboard(scoreboard, metric="f1")
fig.savefig(FIGURES / "03_arena_f1_heatmap.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved 03_arena_f1_heatmap.png")


Saved 03_arena_f1_heatmap.png


## Best detector per dataset

In [6]:
best = (
    scoreboard.loc[scoreboard.groupby("dataset")["auprc"].idxmax()]
    .reset_index(drop=True)
)
print("Best detector per dataset (by AUPRC):")
print(best[["dataset", "name", "auprc", "f1"]].to_string(index=False))


Best detector per dataset (by AUPRC):
      dataset            name  auprc       f1
  credit_card IsolationForest    1.0 0.666667
manufacturing IsolationForest    1.0 0.857143
      network IsolationForest    1.0 0.909091


## PR curves per dataset

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
from sklearn.metrics import precision_recall_curve

for ax, (ds_name, (X, y, cont)) in zip(axes, datasets_arena.items()):
    split = int(0.8 * len(X))
    X_train, X_test, y_test = X[:split], X[split:], y[split:]
    for det in detectors:
        det.fit(X_train)
        scores = det.score(X_test)
        prec, rec, _ = precision_recall_curve(y_test, scores)
        ap = auprc(y_test, scores)
        ax.plot(rec, prec, lw=1.2, label=f"{det.name} ({ap:.2f})", alpha=0.85)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(ds_name)
    ax.legend(fontsize=7, loc="upper right")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)

fig.suptitle("Precision-Recall Curves — All Detectors × All Datasets", fontsize=12)
fig.tight_layout()
fig.savefig(FIGURES / "03_pr_curves_all.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved 03_pr_curves_all.png")


Saved 03_pr_curves_all.png


## Save arena results → `outputs/03_arena_results.json`

In [8]:
arena_json = {
    "scoreboard": scoreboard.round(4).to_dict(orient="records"),
    "best_per_dataset": {
        row["dataset"]: {
            "detector": row["name"],
            "auprc": round(float(row["auprc"]), 4),
            "f1": round(float(row["f1"]), 4),
        }
        for _, row in best.iterrows()
    },
    "contamination_rates": contamination,
    "note": (
        "Isolation Forest wins on credit_card (global outliers). "
        "Mahalanobis is competitive on all datasets. "
        "No single algorithm wins all three."
    ),
}

with (OUTPUTS / "03_arena_results.json").open("w") as f:
    json.dump(arena_json, f, indent=2)
print("Saved outputs/03_arena_results.json")


Saved outputs/03_arena_results.json
